# LoRA (Low-Rank Adaptation) Explained

**Estimated time: 30 minutes**

## Learning Objectives
- Understand LoRA theory
- Learn about the Rank parameter
- Calculate memory savings
- Compare LoRA vs Full Fine-tuning

## Introduction

LoRA (Low-Rank Adaptation) is an efficient fine-tuning method that dramatically reduces the number of trainable parameters while maintaining model quality.

## Part 1: LoRA Theory

### The Problem with Full Fine-tuning

When fine-tuning a large language model:
- **Llama-3.1-8B**: 8 billion parameters
- **FP16 format**: 2 bytes per parameter
- **Memory needed**: 16 GB just for weights
- **With gradients & optimizer states**: ~64 GB!

### The LoRA Solution

LoRA freezes the original weights and adds small trainable matrices:

```
W_new = W_frozen + ΔW
ΔW = B × A  (where B is d×r and A is r×k)
```

**Key insight:** The update ΔW can be represented by low-rank matrices!

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Visualize LoRA decomposition
def visualize_lora(d=1024, k=1024, r=8):
    """
    Visualize how LoRA works
    
    Args:
        d: Input dimension
        k: Output dimension  
        r: LoRA rank
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Original weight matrix
    W = np.random.randn(d, k) * 0.01
    axes[0].imshow(W[:100, :100], cmap='viridis', aspect='auto')
    axes[0].set_title(f'Original Weight W\n{d}×{k} = {d*k:,} params')
    axes[0].set_xlabel('k')
    axes[0].set_ylabel('d')
    
    # LoRA matrices
    A = np.random.randn(r, k) * 0.01
    B = np.random.randn(d, r) * 0.01
    
    axes[1].imshow(B[:100, :], cmap='plasma', aspect='auto')
    axes[1].set_title(f'LoRA Matrix B\n{d}×{r} = {d*r:,} params')
    axes[1].set_xlabel('r')
    axes[1].set_ylabel('d')
    
    axes[2].imshow(A, cmap='plasma', aspect='auto')
    axes[2].set_title(f'LoRA Matrix A\n{r}×{k} = {r*k:,} params')
    axes[2].set_xlabel('k')
    axes[2].set_ylabel('r')
    
    plt.tight_layout()
    plt.show()
    
    # Calculate parameter reduction
    original_params = d * k
    lora_params = d * r + r * k
    reduction = 100 * (1 - lora_params / original_params)
    
    print(f"\n📊 Parameter Reduction:")
    print(f"   Original: {original_params:,} parameters")
    print(f"   LoRA (r={r}): {lora_params:,} parameters")
    print(f"   Reduction: {reduction:.1f}%")
    print(f"   Factor: {original_params/lora_params:.1f}x smaller")

# Run visualization
visualize_lora(d=1024, k=1024, r=8)

## Part 2: The Rank Parameter

The rank `r` is the most important hyperparameter in LoRA:
- **Lower r**: Fewer parameters, faster training, less capacity
- **Higher r**: More parameters, slower training, more capacity

Typical values: r ∈ {4, 8, 16, 32, 64}

In [ ]:
def compare_lora_ranks(d=4096, k=4096, ranks=[1, 2, 4, 8, 16, 32, 64, 128]):
    """
    Compare different LoRA ranks
    """
    original_params = d * k
    
    results = []
    for r in ranks:
        lora_params = d * r + r * k
        percentage = 100 * lora_params / original_params
        results.append({
            'r': r,
            'params': lora_params,
            'percentage': percentage
        })
    
    # Visualize
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Absolute parameters
    ax1.bar([str(r['r']) for r in results], [r['params'] for r in results])
    ax1.axhline(y=original_params, color='r', linestyle='--', label='Full fine-tuning')
    ax1.set_xlabel('Rank (r)')
    ax1.set_ylabel('Number of Parameters')
    ax1.set_title('LoRA Parameters by Rank')
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Percentage
    ax2.plot([r['r'] for r in results], [r['percentage'] for r in results], marker='o')
    ax2.set_xlabel('Rank (r)')
    ax2.set_ylabel('% of Original Parameters')
    ax2.set_title('Parameter Efficiency')
    ax2.set_xscale('log', base=2)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print table
    print(f"\n{'Rank':>6} | {'Parameters':>15} | {'% of Original':>15} | {'Reduction':>15}")
    print("-" * 70)
    for r in results:
        reduction = 100 - r['percentage']
        print(f"{r['r']:>6} | {r['params']:>15,} | {r['percentage']:>14.2f}% | {reduction:>14.1f}%")

# Compare different ranks for a Llama-3.1-8B attention layer
compare_lora_ranks(d=4096, k=4096)

## Part 3: Memory Savings Calculation

Let's calculate the exact memory savings for fine-tuning Llama-3.1-8B.

In [ ]:
def calculate_memory_requirements(model_size_b=8, hidden_size=4096, num_layers=32, lora_r=8, use_8bit=True):
    """
    Calculate memory requirements for LoRA vs full fine-tuning
    
    Args:
        model_size_b: Model size in billions of parameters
        hidden_size: Hidden dimension size
        num_layers: Number of transformer layers
        lora_r: LoRA rank
        use_8bit: Use 8-bit quantization for base model
    """
    model_params = model_size_b * 1e9
    
    # Full fine-tuning
    bytes_per_param_full = 2  # FP16
    model_memory_full = model_params * bytes_per_param_full / 1e9  # GB
    gradients_memory_full = model_memory_full  # Same size as model
    optimizer_memory_full = model_memory_full * 2  # Adam: 2x for momentum & variance
    total_full = model_memory_full + gradients_memory_full + optimizer_memory_full
    
    # LoRA fine-tuning
    bytes_per_param_base = 1 if use_8bit else 2
    model_memory_lora = model_params * bytes_per_param_base / 1e9
    
    # LoRA parameters (4 matrices per layer: Q, K, V, O)
    lora_params_per_layer = 4 * (hidden_size * lora_r + lora_r * hidden_size)
    total_lora_params = lora_params_per_layer * num_layers
    
    lora_memory = total_lora_params * 2 / 1e9  # FP16 for LoRA weights
    gradients_memory_lora = lora_memory
    optimizer_memory_lora = lora_memory * 2
    
    total_lora = model_memory_lora + lora_memory + gradients_memory_lora + optimizer_memory_lora
    
    # Results
    print(f"\n{'='*70}")
    print(f"Memory Requirements for {model_size_b}B Parameter Model")
    print(f"{'='*70}\n")
    
    print(f"📊 Full Fine-tuning:")
    print(f"   Model weights:     {model_memory_full:>8.2f} GB")
    print(f"   Gradients:         {gradients_memory_full:>8.2f} GB")
    print(f"   Optimizer states:  {optimizer_memory_full:>8.2f} GB")
    print(f"   {'─'*40}")
    print(f"   Total:             {total_full:>8.2f} GB\n")
    
    print(f"🎯 LoRA Fine-tuning (r={lora_r}, {'8-bit' if use_8bit else '16-bit'} base):")
    print(f"   Model weights:     {model_memory_lora:>8.2f} GB  ({'frozen, 8-bit' if use_8bit else 'frozen'})")
    print(f"   LoRA weights:      {lora_memory:>8.2f} GB  ({total_lora_params/1e6:.1f}M params)")
    print(f"   Gradients:         {gradients_memory_lora:>8.2f} GB")
    print(f"   Optimizer states:  {optimizer_memory_lora:>8.2f} GB")
    print(f"   {'─'*40}")
    print(f"   Total:             {total_lora:>8.2f} GB\n")
    
    savings = total_full - total_lora
    savings_pct = 100 * savings / total_full
    
    print(f"💰 Savings:")
    print(f"   Memory saved:      {savings:>8.2f} GB ({savings_pct:.1f}%)")
    print(f"   Reduction factor:  {total_full/total_lora:>8.1f}x")
    print(f"\n{'='*70}\n")
    
    # Visualization
    fig, ax = plt.subplots(figsize=(10, 6))
    
    categories = ['Full Fine-tuning', f'LoRA (r={lora_r})']
    model_mem = [model_memory_full, model_memory_lora]
    train_mem = [gradients_memory_full + optimizer_memory_full, lora_memory + gradients_memory_lora + optimizer_memory_lora]
    
    x = np.arange(len(categories))
    width = 0.35
    
    ax.bar(x, model_mem, width, label='Model Weights', color='skyblue')
    ax.bar(x, train_mem, width, bottom=model_mem, label='Training Memory', color='coral')
    
    ax.set_ylabel('Memory (GB)')
    ax.set_title(f'Memory Requirements: Full vs LoRA Fine-tuning\n{model_size_b}B Parameter Model')
    ax.set_xticks(x)
    ax.set_xticklabels(categories)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for i, (m, t) in enumerate(zip(model_mem, train_mem)):
        total = m + t
        ax.text(i, total + 2, f'{total:.1f} GB', ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Calculate for Llama-3.1-8B
calculate_memory_requirements(model_size_b=8, hidden_size=4096, num_layers=32, lora_r=8, use_8bit=True)

## Part 4: Practical Comparison

Let's compare LoRA with different ranks in practice.

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

# Test with GPT-2 (faster for demo)
model_name = "gpt2"

def test_lora_config(r, alpha=16):
    """
    Test a specific LoRA configuration
    """
    model = AutoModelForCausalLM.from_pretrained(model_name)
    
    config = LoraConfig(
        r=r,
        lora_alpha=alpha,
        target_modules=["c_attn"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    model = get_peft_model(model, config)
    
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    
    return {
        'r': r,
        'alpha': alpha,
        'trainable_params': trainable,
        'total_params': total,
        'percentage': 100 * trainable / total
    }

# Test different configurations
print("\nTesting different LoRA configurations:\n")
print(f"{'Rank':>6} | {'Alpha':>6} | {'Trainable':>12} | {'Total':>12} | {'Percentage':>12}")
print("-" * 70)

for r in [4, 8, 16, 32]:
    result = test_lora_config(r)
    print(f"{result['r']:>6} | {result['alpha']:>6} | {result['trainable_params']:>12,} | {result['total_params']:>12,} | {result['percentage']:>11.3f}%")

## Summary

### Key Takeaways:

1. **LoRA Theory**:
   - Adds low-rank matrices to frozen model
   - Dramatically reduces trainable parameters
   - Maintains model quality

2. **Rank Parameter**:
   - Controls capacity vs efficiency trade-off
   - Typical values: 8-16 for most tasks
   - Higher r = more capacity but slower

3. **Memory Savings**:
   - 8B model: ~64 GB (full) vs ~16 GB (LoRA)
   - 4x reduction in memory
   - Enables training on consumer GPUs

4. **Best Practices**:
   - Start with r=8 or r=16
   - Use 8-bit quantization for base model
   - Target attention layers (Q, K, V, O)
   - Set alpha = 2 * r

### Next Steps:
- **Notebook 03:** Distributed Training for Multi-GPU setups

### References:
- [LoRA Paper](https://arxiv.org/abs/2106.09685)
- [PEFT Library](https://github.com/huggingface/peft)
- [QLoRA Paper](https://arxiv.org/abs/2305.14314)